# MaduraApp — Fine-tuning YOLO26n en Google Colab

Pipeline CRISP-DM de entrenamiento del modelo de madurez agrícola.

**Antes de ejecutar cualquier celda:**
1. Menú superior → `Runtime` → `Change runtime type` → **GPU → T4**
2. Tener el archivo `maduraapp_dataset.zip` subido a tu Google Drive
   (ver Paso 0 en la guía)

| Celda | Descripción | Tiempo aprox. |
|-------|-------------|---------------|
| 1 | Verificar GPU | < 1 min |
| 2 | Instalar dependencias | 2–3 min |
| 3 | Montar Drive + descomprimir | 5–10 min |
| 4 | Verificar dataset | < 1 min |
| 5 | Generar data.yaml | < 1 min |
| 6 | **Entrenar (80 épocas)** | **2–4 horas** |
| 7 | Evaluar mAP\@50 | 5–10 min |
| 8 | Guardar best.pt en Drive | < 1 min |
| 9 | Descargar best.pt al PC | < 1 min |

## Celda 1 — Verificar GPU

Debe mostrar `True` y el nombre de la GPU (T4 en Colab gratuito).

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU     : {gpu} ({mem:.1f} GB VRAM)')
else:
    print('GPU: NO DISPONIBLE — ve a Runtime → Change runtime type → GPU')
    raise SystemExit('Detener: necesitas GPU para entrenar YOLO26n')

## Celda 2 — Instalar dependencias

In [ ]:
!pip install -q ultralytics pyyaml

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics {ultralytics.__version__} instalado')

## Celda 3 — Montar Google Drive y descomprimir dataset

Se pedirá autorización para acceder a tu Drive (ventana emergente).

**Ruta esperada en Drive:** `Mi unidad/maduraapp_dataset.zip`  
Si la subiste a otra carpeta, cambia `DRIVE_ZIP_PATH` abajo.

In [ ]:
from google.colab import drive
import pathlib, subprocess, time

# ── Montar Drive ──────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Configuración de rutas (ajustar si cambiaste la ubicación del ZIP) ────
DRIVE_ZIP_PATH = '/content/drive/MyDrive/maduraapp_dataset.zip'
LOCAL_DATASET  = '/content/maduraapp'

# ── Verificar que el ZIP existe ───────────────────────────────────────────
if not pathlib.Path(DRIVE_ZIP_PATH).exists():
    raise FileNotFoundError(
        f'No se encontró {DRIVE_ZIP_PATH}\n'
        'Asegúrate de haber subido maduraapp_dataset.zip a la raíz de tu Drive.'
    )
zip_size = pathlib.Path(DRIVE_ZIP_PATH).stat().st_size / 1e9
print(f'ZIP encontrado: {zip_size:.2f} GB')

# ── Descomprimir al disco local de Colab (más rápido para I/O de training) 
if not pathlib.Path(LOCAL_DATASET).exists():
    print('Descomprimiendo dataset (puede tardar ~5 min)...')
    t0 = time.time()
    result = subprocess.run(
        ['unzip', '-q', DRIVE_ZIP_PATH, '-d', '/content/'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if result.returncode != 0:
        print('Error al descomprimir:', result.stderr[:500])
    else:
        print(f'Dataset descomprimido en {elapsed:.0f}s → {LOCAL_DATASET}')
else:
    print(f'Dataset ya existe en {LOCAL_DATASET} — reutilizando.')

## Celda 4 — Verificar dataset

Confirma que las 12 clases tienen imágenes y que los labels son válidos.

In [ ]:
import pathlib

CLASSES = [
    'aguacate_hass_INMADURO', 'aguacate_hass_OPTIMO', 'aguacate_hass_SOBRE_MADURO',
    'platano_INMADURO', 'platano_OPTIMO', 'platano_SOBRE_MADURO',
    'tomate_usda_INMADURO', 'tomate_usda_OPTIMO', 'tomate_usda_SOBRE_MADURO',
    'mango_INMADURO', 'mango_OPTIMO', 'mango_SOBRE_MADURO',
]

base = pathlib.Path(LOCAL_DATASET)
print(f'{"Split":<8}  {"Imágenes":>9}  {"Labels":>9}')
print('-' * 32)
total_imgs = 0
for split in ('train', 'valid', 'test'):
    imgs   = len(list((base / split / 'images').glob('*')))
    labels = len(list((base / split / 'labels').glob('*.txt')))
    print(f'{split:<8}  {imgs:>9,}  {labels:>9,}')
    total_imgs += imgs
print('-' * 32)
print(f'{"TOTAL":<8}  {total_imgs:>9,}')

# Verificar distribución de class_ids en train
from collections import Counter
counts = Counter()
for txt in (base / 'train' / 'labels').glob('*.txt'):
    for line in txt.read_text().splitlines():
        parts = line.strip().split()
        if parts:
            counts[int(parts[0])] += 1

print()
print(f'{"class_id":<5} {"clase":<35} {"bboxes":>8}')
print('-' * 52)
for cid, name in enumerate(CLASSES):
    n = counts.get(cid, 0)
    warn = ' ⚠' if n < 500 else ''
    print(f'{cid:<5} {name:<35} {n:>8,}{warn}')

## Celda 5 — Generar data.yaml

Crea el archivo de configuración del dataset apuntando a las rutas de Colab.

In [ ]:
import yaml

DATA_YAML = f'{LOCAL_DATASET}/data.yaml'

data_cfg = {
    'path':  LOCAL_DATASET,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    12,
    'names': CLASSES,
}

with open(DATA_YAML, 'w') as fh:
    yaml.dump(data_cfg, fh, allow_unicode=True)

print('data.yaml generado:')
print(open(DATA_YAML).read())

## Celda 6 — Entrenamiento YOLO26n (80 épocas)

⏱️ **Tiempo estimado: 2–4 horas** en GPU T4 (Colab gratuito).

- Los resultados y checkpoints se guardan en `runs/maduraapp_v1/`
- Si la sesión se corta, vuelve a ejecutar esta celda — Ultralytics reanuda desde el último checkpoint si existe `last.pt`
- El batch de 16 es el óptimo para T4 (16 GB VRAM). Si hay OOM, bájalo a 8.

In [ ]:
from ultralytics import YOLO
import pathlib

# Reanudar entrenamiento si existe un checkpoint previo
RESUME_PT = pathlib.Path('runs/maduraapp_v1/weights/last.pt')

if RESUME_PT.exists():
    print(f'Checkpoint encontrado en {RESUME_PT} — reanudando entrenamiento...')
    model = YOLO(str(RESUME_PT))
    results = model.train(resume=True)
else:
    print('Iniciando entrenamiento desde cero con YOLO26n...')
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=DATA_YAML,
        epochs=80,
        batch=16,
        imgsz=640,
        device=0,           # GPU 0
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        cos_lr=True,
        amp=True,
        patience=15,
        # Augmentation (igual que scripts/config.yaml)
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=15.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        # Reporting
        project='runs',
        name='maduraapp_v1',
        save_period=10,
        plots=True,
        exist_ok=True,
    )

print('\nEntrenamiento finalizado.')
print(f'Resultados en: runs/maduraapp_v1/')

## Celda 7 — Evaluación sobre el set de test

KPI objetivo: **mAP\@50 ≥ 0.75**  
Si no se alcanza, revisa la tabla de tuning en `scripts/README.md`.

In [ ]:
from ultralytics import YOLO

BEST_PT = 'runs/maduraapp_v1/weights/best.pt'
model   = YOLO(BEST_PT)
metrics = model.val(data=DATA_YAML, split='test', plots=True)

map50    = metrics.box.map50
map5095  = metrics.box.map
prec     = metrics.box.mp
recall   = metrics.box.mr

print()
print('=' * 45)
print(' RESULTADOS FINALES — MaduraApp v1')
print('=' * 45)
print(f'  mAP@50    = {map50:.4f}   (target ≥ 0.75)')
print(f'  mAP@50-95 = {map5095:.4f}')
print(f'  Precision = {prec:.4f}')
print(f'  Recall    = {recall:.4f}')
print()
TARGET = 0.75
if map50 >= TARGET:
    print(f'  KPI APROBADO — modelo listo para deploy')
else:
    print(f'  KPI NO ALCANZADO (delta = {map50 - TARGET:.3f})')
    print('  Sugerencias:')
    print('    - Extender a 120 epocas:  model.train(epochs=120, resume=True)')
    print('    - Reducir lr0 a 0.0005 y reentrenar')
    print('    - Agregar mas imagenes en las clases con bajo recall')
print('=' * 45)

# Mostrar mAP por clase
print()
print('mAP@50 por clase:')
for i, (name, ap) in enumerate(zip(CLASSES, metrics.box.ap50)):
    bar = '#' * int(ap * 20)
    print(f'  {i:>2} {name:<35} {ap:.3f} {bar}')

## Celda 8 — Guardar best.pt en Google Drive (backup)

El runtime de Colab se borra al cerrar la sesión. Guarda el modelo en Drive para no perderlo.

In [ ]:
import shutil, pathlib

# Guardar en Drive
DRIVE_BACKUP = '/content/drive/MyDrive/maduraapp_best.pt'
shutil.copy2(BEST_PT, DRIVE_BACKUP)
size_mb = pathlib.Path(DRIVE_BACKUP).stat().st_size / 1e6
print(f'best.pt guardado en Drive: {DRIVE_BACKUP} ({size_mb:.1f} MB)')

# También copiar la carpeta de runs completa (curvas, confusion matrix, etc.)
DRIVE_RUNS = '/content/drive/MyDrive/maduraapp_runs_v1'
if not pathlib.Path(DRIVE_RUNS).exists():
    shutil.copytree('runs/maduraapp_v1', DRIVE_RUNS)
    print(f'Carpeta de resultados guardada en: {DRIVE_RUNS}')
else:
    print(f'Ya existe {DRIVE_RUNS} — omitido (borrar manualmente para sobreescribir)')

## Celda 9 — Descargar best.pt al PC

Descarga el modelo al PC. Luego muévelo a `backend/weights/` con:
```bash
python scripts/export_model.py
```

In [ ]:
from google.colab import files
files.download(BEST_PT)
print(f'Descargando {BEST_PT}...')
print('Una vez descargado, en tu PC ejecuta:')
print('  cp ~/Downloads/best.pt runs/maduraapp_v1/weights/best.pt')
print('  python scripts/export_model.py')

## (Opcional) Celda 10 — Visualizar predicciones sobre imágenes de test

Muestra 5 imágenes aleatorias del set de test con las predicciones del modelo.

In [ ]:
import glob, random
from IPython.display import Image, display
from ultralytics import YOLO

model = YOLO(BEST_PT)
test_imgs = glob.glob(f'{LOCAL_DATASET}/test/images/*.jpg')
samples   = random.sample(test_imgs, min(5, len(test_imgs)))

for img_path in samples:
    result = model(img_path)[0]
    out_path = result.save()
    print(f'\n{img_path.split("/")[-1]}')
    display(Image(out_path))